# Phase 3: Identify Usable Resolved Conversations

### Objective
Partition the reconstructed conversation threads into three strict quality tiers:
1. **`usable`**: Clear customer problem + verified public resolution (explicit customer confirmation or concrete completed brand action). Final turn is NOT a deflection. **Only these are indexed for RAG retrieval in Phase 7-8.**
2. **`partially_usable`**: Clear customer problem + brand engagement, but ends in DM deflection, order lookup request, or unanswered customer reply. **Used for intent clustering in Phase 4-5, but strictly excluded from retrieval index.**
3. **`unusable`**: Single orphaned tweets, spam, garbled text, or pure deflection with zero problem context.


In [ ]:
import os
import sys
import json
sys.path.append(os.path.abspath("../src"))

from data_processing import process_and_classify_threads, save_classified_dataset

threads_path = "../data/processed/amazon_threads.json"
with open(threads_path, "r", encoding="utf-8") as f:
    threads = json.load(f)

print(f"Loaded {len(threads):,} reconstructed threads.")
classified_buckets = process_and_classify_threads(threads)
save_classified_dataset(classified_buckets, output_dir="../data/processed")

In [ ]:
# Summary distribution
total = len(threads)
print("=== Thread Classification Distribution ===")
for category, items in classified_buckets.items():
    pct = (len(items) / total) * 100
    print(f"• {category.upper():<16}: {len(items):>6,} ({pct:.1f}%)")

In [ ]:
# Inspect 10 examples per category
for cat in ["usable", "partially_usable", "unusable"]:
    print(f"\n{'='*30}\n10 EXAMPLES: {cat.upper()}\n{'='*30}")
    for i, thread in enumerate(classified_buckets[cat][:10]):
        print(f"\n[{cat.upper()} #{i+1}] ID: {thread['thread_id']} | Reason: {thread['quality_reason']}")
        for msg in thread['messages']:
            print(f"   [{msg['speaker'].upper()}]: {msg['text']}")